### Observables
Wir nennen ein Python Objekt ein **Observable**, falls es eine Methode `observe(fun: function)`zur Verfügung stellt,
die es erlaubt Funktion `fun` als Callbacks registrieren zu können, und über eine private Method
`_notify(event: str, **kwargs: Any)` verfügt, die bei einer
Zustandsänderung, z.B. wenn bei einem Spiel ein Stein verschoben wird, 
`_notify('push_stone', old=(1, 1), new=(2, 3))`
aufruft, die dann alle  registrierten Callbacks `fun` mit den
Argumenten `fun('push_stone', old=(1, 1), new=(2, 3))` aufruft.

Man kann sich das Text-Widget als *Observable* denken, welches zusätzlich erlaubt
festzulegen, bei welchen Zustandsänderungen das Callback aufgerufen wird.

In [ ]:
from ipywidgets import Text, Output
from IPython.display import display

out = Output(layout={'border': '1px solid black'})
eingabe = Text(placeholder='type something', continuous_update=False)


@out.capture()
def show_change(change):
    if change.new == '':
        return
    out.clear_output()
    print(change)
    change.owner.value = ''


eingabe.observe(show_change, names='value')
display(eingabe, out)
eingabe.focus()

### Game-Objekt als *Observable*
Ein `Game`-Objekt kann man sich als *Observable* implementieren. 
Ändert der Spielzustand, werden dann die von einem für die Darstellung verantwortlichen `View`-Objekt
registrierten Callbacks aufgerufen, deren Argumente auf einfache Weise das
updaten der graphischen Darstellung des Spielzustandes erlauben.

Das `View`-Objekt hat in der Regel auf Zugriff auf das Game-Objekt.

Nachstehend implementieren wir ein Spiel als *Observable*. Der Spielzustand speichert die Position eines Steins in einem Gitter. Der Spieler kann mit
- `new_game()` ein neues Spiel starten. Ein Counter wird erhöht.
  Registriete Callbacks `f` werden mit `f(new_game, counter)` aufgerufen,
- `push_stone((dx, dy))` von der aktuellen Positon `(x, y)` nach `(x+dx, y+dy)` verschieben.
- Registriete Callbacks `f` werden mit `f('push_stone', (x, y), (x+dx, y+dy))` aufgerufen.

Um das Debuggen zu erleichern, definieren wir auch die spezielle Methode
`__repr__`, welche ein String-Repräsentation des `Game`-Objektes liefert, welche
den Spielzustand textlich wiedergibt.

Eine  View-Komponente kann nun eine Funktion als Callback registrieren, die in diesem Fall
das Feld `(x, y)` leert und den Stein neu
auf dem Feld `(x+dx, y+dy)` des Gitters zeichnet.

In [ ]:
class Game:
    def __init__(self):
        self.callbacks = {}
        self.grid_size = 8
        self.games_played = 0
        self.pos = (4, 4)

    def observe(self, fun):
        self.callbacks[fun.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def _notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)

    def new_game(self):
        self.games_played += 1
        self.pos = (4, 4)
        self._notify('new_game', games_played=self.games_played)

    def is_inside(self, pos):
        return (0 <= pos[0] < self.grid_size
                and 0 <= pos[1] < self.grid_size)

    def push_stone(self, dpos):
        dx, dy = dpos
        x, y = self.pos
        npos = (x+dx, y + dy)

        if not self.is_inside(npos):
            self._notify('error',
                         msg=f'cannot push stone from  {self.pos} to {npos}!',
                         )
            return

        self.pos = npos
        self._notify('push_stone', old=(x, y), new=npos)

    def __repr__(self):
        return f'Games played: {self.games_played}, grid size: {self.grid_size},\n'\
               f'pos: {self.pos}'

In [ ]:
game = Game()
game

In [ ]:
game.push_stone((1, 2))
game

In [ ]:
# Callback registrieren, das meldet, was zu tun ist


def report(event, **kwargs):
    print(event, kwargs)

In [ ]:
game.observe(report)

In [ ]:
game.new_game()
game

In [ ]:
game.push_stone((6, 6))
game

In [ ]:
game.push_stone((2, 3))
game

In [ ]:
import canvas_helpers as H
from ipycanvas import MultiCanvas
from ipywidgets import Output
from IPython.display import display


class View:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec

        self.out = Output(layout={'border': '1px solid black'})
        self.mcanvas = MultiCanvas(2, width=200, height=200, layout={'border': '1px solid black'})
        self.bg, self.fg = self.mcanvas
        H.draw_grid(self.fg, boardspec, line_width=3, color='blue')

        self.game.observe(self.update)
        self.game.new_game()

    def new_game(self, **kwargs):
        self.bg.clear()
        H.fill_field(self.bg, self.game.pos, self.boardspec, color='red')
        self.out.append_stdout(f'Round {kwargs['games_played']} started at {self.game.pos}\n')

    def push_stone(self, **kwargs):
        H.clear_field(self.bg, kwargs['old'], self.boardspec)
        H.fill_field(self.bg, kwargs['new'], self.boardspec, color='red')

    def error(self, **kwargs):
        self.out.append_stdout(f'ERROR: {kwargs['msg']}\n')

    def update(self, event,  **kwargs):
        self.out.append_stdout(f'{event}, {kwargs}\n')
        if hasattr(self, event):
            getattr(self, event)(**kwargs)

    def _ipython_display_(self):
        display(self.mcanvas, self.out)

In [ ]:
game = Game()
view = View(game)
game.observe(view.update)
view

In [ ]:
game.new_game()

In [ ]:
game.push_stone((3, 3))
game.push_stone((1, -1))
game

### Das Model-View-Controller Pattern
Neben der Spielkomponente, dem Modell,  und der View,
betrachtet man  oft noch eine Controller-Komponente.
Diese nimmt Benutzerinput entgegen, interpretiert diesen und ruft dann die
entsprechenden Funktionen der Game-Komponente auf.
Jedoch sind View und Controller oft eng verschmolzen. Z.B ist ein
Canvas-Widget gleichermassen für das Entgegennehmen von Tasten-und Mausevents, wie auf für das Zeichnen zuständig.

Wir benutzen nachstehend ein kleines Canvas-Widget als Controller. Es hört auf Tastendrücke. Beim Drücken von `n` wird ein neues Spiel startet und
beim Drücken einer Pfeiltaste die Funktion `push_stone` mit dem entsprechenden Argument aufrufen.

In [ ]:
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display

# nur waehrend der Entwicklungsphase
# err_out = Output(layout={'border': '1px solid black'})


class C:
    def __init__(self, game):
        self.game = game
        self.canvas = Canvas(width=20, height=20, layout={'border': '1px solid black'})
        self.canvas.on_key_down(self.on_key_down)
 
    # @err_out.capture(clear_output=True)
    def on_key_down(self, key, *flags):
        key_direction = {'ArrowUp': (0, -1),
                         'ArrowDown': (0, 1),
                         'ArrowRight': (1, 0),
                         'ArrowLeft': (-1, 0),
                         }
        if key in key_direction:
            dpos = key_direction[key]
            game.push_stone(dpos)
        elif key == 'n':
            game.new_game()

    def _ipython_display_(self):
        # display(self.canvas, err_out)
        display(self.canvas)
        self.canvas.focus()

In [ ]:
c = C(game)
view = View(game)
display(c, view)